In [2]:
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

# 1. Load the dataset
df = sns.load_dataset('titanic')
df

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True
888,0,3,female,NaN,1,2,23.4500,S,Third,woman,False,NaN,Southampton,no,False
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True


In [4]:
# selecting useful columns only
features = ['pclass', 'sex', 'age', 'fare', 'sibsp', 'parch', 'embarked']
X = df[features]
y = df['survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train.isnull().sum()

,0
pclass,0
sex,0
age,140
fare,0
sibsp,0
parch,0
embarked,2


In [6]:
# now i have to fill age i can use that column transformer for that
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# fill missing values with median
numeric_transformer = SimpleImputer(strategy='median')
categorical_transformer = OneHotEncoder(drop='first', sparse_output=False)

#  i am combining them into one Column Transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, ['age', 'fare', 'sibsp', 'parch', 'pclass']),
        ('cat', categorical_transformer, ['sex', 'embarked'])
    ],
    remainder = 'passthrough'
)

preprocessor

ColumnTransformer(remainder='passthrough',
                  transformers=[('num', SimpleImputer(strategy='median'),
                                 ['age', 'fare', 'sibsp', 'parch', 'pclass']),
                                ('cat',
                                 OneHotEncoder(drop='first',
                                               sparse_output=False),
                                 ['sex', 'embarked'])])

# now i will chain this preprocessor directly to random forest classifier

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

master_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# train the entire pipeline with one line of code!
master_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num',
                                                  SimpleImputer(strategy='median'),
                                                  ['age', 'fare', 'sibsp',
                                                   'parch', 'pclass']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first',
                                                                sparse_output=False),
                                                  ['sex', 'embarked'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

In [8]:
# it is very easy now to check
from sklearn.metrics import accuracy_score
y_pred = master_pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Titanic Survival Prediction Accuracy: {accuracy * 100:.2f}%")

Titanic Survival Prediction Accuracy: 82.68%


In [9]:
X_test

,pclass,sex,age,fare,sibsp,parch,embarked
709,3,male,NaN,15.2458,1,1,C
439,2,male,31.0,10.5000,0,0,S
840,3,male,20.0,7.9250,0,0,S
720,2,female,6.0,33.0000,0,1,S
39,3,female,14.0,11.2417,1,0,C
...,...,...,...,...,...,...,...
433,3,male,17.0,7.1250,0,0,S
773,3,male,NaN,7.2250,0,0,C
25,3,female,38.0,31.3875,1,5,S
84,2,female,17.0,10.5000,0,0,S


In [12]:
# let's check how our model reacts with production level
import numpy as np

production_data = pd.DataFrame({
    'pclass': [1,2],
    'sex':['male', 'female'],
    'age':[np.nan, 24],
    'fare':[85.67, 4.33],
    'sibsp':[1,2],
    'parch':[1,0],
    'embarked':['C','S']
})

display(production_data)
production_predictions = master_pipeline.predict(production_data)
print("\nPredictions (0 = Did not survive, 1 = Survived):")
print(production_predictions)

,pclass,sex,age,fare,sibsp,parch,embarked
0,1,male,NaN,85.67,1,1,C
1,2,female,24.0,4.33,2,0,S



Predictions (0 = Did not survive, 1 = Survived):
[0 1]


# saving for real production

In [13]:
import joblib

joblib.dump(master_pipeline, 'titanic_survival_model.pkl')

print("Model successfully saved to disk!")

Model successfully saved to disk!
